# Create Embeddings

## What this notebook does

* **Initialize a reproducible “embeddings run” context** using `infra.init_embeddings_run(...)`: selects the O*NET release (`ONET_VERSION`), embedding encoder (`ENCODER_NAME`), and year, then creates or reuses a deterministic `RUN_TAG` and corresponding run directory structure (`exports/`, `figures/`, `logs/`, `cache/`). The active run is stored globally in `infra.RP`, and key provenance fields (paths, encoder ID, run diff/status) are printed.  

* **Set notebook-level reproducibility knobs** (`SEED`, `BATCH_SIZE`) and seed NumPy for any downstream randomized steps (even though embedding itself is deterministic given inputs/provider).

* **Load O*NET data via a read-only API** (`onet.get_db`, `onet.build_df_tasks`, `onet.load_occ_meta`), optionally including supplemental tasks, and perform a basic duplicate guard on `(onet_code, Task ID)`. The resulting task-level dataframe (`df_tasks`) and occupation metadata (`df_occ_meta`) are exported to the *current run*’s `exports/` folder as CSV.  

* **Compute or load task text embeddings with strict run persistence**:

  * Extracts the list of task statements (`df_tasks["Task"]`).
  * Calls `load_or_compute_run_embeddings(...)` to either:

    * load an existing embedding matrix from `exports/task_embeddings.npy` if its fingerprint matches, or
    * compute embeddings in batches using the selected `EncoderSpec`, leveraging a **global per-text cache** (`cache_scope="global"`) to avoid recomputation across runs, then write the full matrix and a fingerprint JSON to the current run exports.
  * Validates that the embeddings match the saved fingerprint (`validate_run_embeddings(...)`) and prints shape + embedder identity/signature.  


In [1]:
# === Cell 1: Init embeddings run (non-interactive; infra owns the logic) ===

import importlib
import infra
import embeddings

importlib.reload(infra)
importlib.reload(embeddings)

# Notebook defaults
YEAR = 2025
ONET_VERSION = "30_1"           # accepts "30_0" or "30.0" downstream
ENCODER_NAME = "openai-3-large" # Selectable models: 
                                #    openai-3-large (requires OpenAI Key)
                                #    st-minilm-l6-v2
                                #    st-gtr-t5-large
                                #    mistral-embed-norm
NOTE = ""

RP, cfg, ENCODER_SPEC, st = infra.init_embeddings_run(
    year=YEAR,
    onet_version=ONET_VERSION,
    encoder_name=ENCODER_NAME,
    prefix="embeddings",
    note=NOTE,
    persist=True,
    ensure_dirs=True,
    return_status=True,  # drop if you don't want status
)

print("RUN_ACTION   :", st.action)
print("RUN_NOTE     :", st.note)
if st.diff:
    print("RUN_DIFF     :", st.diff)

# Notebook policy
SEED = 42
BATCH_SIZE = 256

import numpy as np
np.random.seed(SEED)

print("PROJECT_ROOT :", infra.PROJECT_ROOT)
print("RUN_TAG      :", infra.RUN_TAG)
print("RUN_DIR      :", infra.RUN_DIR)
print("Exports      :", RP.exports)
print("Cache (run)  :", RP.cache)
print("Cache (glob) :", infra.GLOBAL_CACHE_ROOT)
print("YEAR         :", int(cfg.get("year", YEAR)))
print("ONET_VERSION :", str(cfg.get("onet_version", ONET_VERSION)))
print("ENCODER_NAME :", str(cfg.get("encoder_name", ENCODER_NAME)))
print("ENCODER_ID   :", ENCODER_SPEC.embedder_id())
print("SEED         :", SEED)
print("BATCH_SIZE   :", BATCH_SIZE)


RUN_ACTION   : reuse
RUN_NOTE     : reused matching base run
PROJECT_ROOT : /home/joc/code/geometry-of-work
RUN_TAG      : embeddings__openai__text-embedding-3-large__d3072__year-2025__v30_1
RUN_DIR      : /home/joc/code/geometry-of-work/out/runs/embeddings__openai__text-embedding-3-large__d3072__year-2025__v30_1
Exports      : /home/joc/code/geometry-of-work/out/runs/embeddings__openai__text-embedding-3-large__d3072__year-2025__v30_1/exports
Cache (run)  : /home/joc/code/geometry-of-work/out/runs/embeddings__openai__text-embedding-3-large__d3072__year-2025__v30_1/cache
Cache (glob) : /home/joc/code/geometry-of-work/out/_cache
YEAR         : 2025
ONET_VERSION : 30_1
ENCODER_NAME : openai-3-large
ENCODER_ID   : openai:text-embedding-3-large?dim=3072
SEED         : 42
BATCH_SIZE   : 256


In [2]:
# === Cell 2: Load O*NET + build df_tasks + save (STRICT current run) ===
import pandas as pd
import infra
import onet

importlib.reload(onet)

# strict: kräver att Cell 1 har aktiverat run
RP = infra.RP
if RP is None:
    raise RuntimeError("No active run. Run Cell 1 first.")

ONET_VERSION = str(cfg.get("onet_version", ONET_VERSION))

# 1) Get O*NET DB (accepts "30_0" or "30.0")
db = onet.get_db(version=ONET_VERSION)

# 2) Build via read-only API (preferred)
INCLUDE_SUPPLEMENTAL = True
df_tasks = onet.build_df_tasks(db, include_supplemental=INCLUDE_SUPPLEMENTAL, require_rle=True)
df_occ_meta = onet.load_occ_meta(db)

# Guard: duplicates
dups = int(df_tasks.duplicated(subset=["onet_code", "Task ID"]).sum())
if dups:
    print(f"⚠️ {dups} duplicates on (onet_code, Task ID).")

print(
    f"✅ O*NET loaded via onet-module:\n"
    f"   version:     {db.version}\n"
    f"   df_occ_meta: {len(df_occ_meta):,}\n"
    f"   df_tasks:    {len(df_tasks):,} (core+supp={INCLUDE_SUPPLEMENTAL})"
)

# 3) Save (strict current run)
fp_tasks = RP.export_fp("tasks_for_pca_base.csv")
fp_occ   = RP.export_fp("occ_meta.csv")

df_tasks.to_csv(fp_tasks, index=False)
df_occ_meta.to_csv(fp_occ, index=False)

print("✅ Saved (current run):")
print(" -", fp_tasks)
print(" -", fp_occ)


✅ O*NET loaded via onet-module:
   version:     30.1
   df_occ_meta: 1,016
   df_tasks:    17,606 (core+supp=True)
✅ Saved (current run):
 - /home/joc/code/geometry-of-work/out/runs/embeddings__openai__text-embedding-3-large__d3072__year-2025__v30_1/exports/tasks_for_pca_base.csv
 - /home/joc/code/geometry-of-work/out/runs/embeddings__openai__text-embedding-3-large__d3072__year-2025__v30_1/exports/occ_meta.csv


In [4]:
# === Cell 3: Compute/load embeddings (STRICT current run) ===
import infra
from embeddings import load_or_compute_run_embeddings, validate_run_embeddings

RP = infra.RP
if RP is None:
    raise RuntimeError("No active run. Run Cell 1 first.")

task_texts = df_tasks["Task"].astype(str).tolist()

emb_npy = RP.export_fp("task_embeddings.npy")
fp_json = RP.export_fp("task_embeddings_fingerprint.json")

task_embeddings = load_or_compute_run_embeddings(
    task_texts,
    spec=ENCODER_SPEC,
    batch_size=BATCH_SIZE,
    emb_npy=emb_npy,
    fp_json=fp_json,
    cache_scope="global",
    allow_none=False,
    show_progress=True,
    progress_desc=f"Embedding ({ENCODER_SPEC.embedder_slug()})",
)

fp = validate_run_embeddings(task_texts, task_embeddings, fp_json=fp_json)

print("✅ Embeddings OK")
print("Shape        :", tuple(task_embeddings.shape))
print("embedder_id  :", fp.get("embedder_id"))
print("embedder_sig :", fp.get("embedder_sig"))
print("Fingerprint  :", fp_json)
print("Embeddings   :", emb_npy)


Embedding (openai_text-embedding-3-large_dim_3072):   0%|          | 0/46 [00:00<?, ?batch/s]

✅ Embeddings OK
Shape        : (17606, 3072)
embedder_id  : openai:text-embedding-3-large?dim=3072
embedder_sig : openai_text-embedding-3-large_dim_3072__prev0.2__a9908cabb6
Fingerprint  : /home/joc/code/geometry-of-work/out/runs/embeddings__openai__text-embedding-3-large__d3072__year-2025__v30_1/exports/task_embeddings_fingerprint.json
Embeddings   : /home/joc/code/geometry-of-work/out/runs/embeddings__openai__text-embedding-3-large__d3072__year-2025__v30_1/exports/task_embeddings.npy
